In [ ]:
import numpy as np
import os
import os.path as op
import pandas as pd
from alternet.splicefactor_evidence import *
from alternet.compare_nets import *

from alternet.gtex_dataloader import *
from alternet.annotation import *

data_path = "/data/bionets/og86asub/alternet-project/alternet/data"
results_path = "/data/bionets/og86asub/alternet-project/alternet/raw_networks/"

# Reference files
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"
tf_list_path = "allTFs_hg38.txt"
sf_list_path = "splicefactors.csv"

# Expression data
gtex_transcript_tpm_path = "GTEx_Analysis_v10_RSEMv1.3.3_transcripts_tpm.txt"
gtex_sample_attributes_path = "GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt"

# Tissue to analyze

CONDITION = TISSUE

# Number of GRNBoost2 runs
N_RUNS = 1

os.makedirs(results_path, exist_ok=True)


biomart = pd.read_csv(op.join(data_path, biomart_path), sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()
appris_df = pd.read_csv(op.join(data_path,appris_path), sep='\t')
digger_df = pd.read_csv(op.join(data_path,digger_path), low_memory=False)
# Load and map TF list
tf_list_raw = pd.read_csv(op.join(data_path,tf_list_path), sep='\t', header=None)
tf_list = map_tf_ids(tf_list_raw, biomart)
# Load and map SF list
sf_list_raw = pd.read_csv(op.join(data_path, sf_list_path), header=0, sep = ',')
sf_list = map_sf_ids(sf_list_raw.loc[:, ['Splicing_Factor']], biomart)
# Combine TF and SF lists
regulator_list = combine_tf_sf_lists(tf_list, sf_list)
tx_to_regtype = dict(zip(regulator_list['Transcript stable ID'], regulator_list['Regulator_type']))
gene_to_regtype = regulator_list.groupby('Gene stable ID')['Regulator_type'].first().to_dict()

VARIANCE_PERCENTILE = 0.7  # Keep top 30%

In [6]:
# gtex_data_dir = '/data/bionets/datasets/hackathon/data/GTEX'

# tissues = ['Blood', 'Brain', 'Adipose Tissue', 'Muscle', 'Blood Vessel',
#        'Heart', 'Ovary', 'Uterus', 'Vagina', 'Breast', 'Skin',
#        'Salivary Gland', 'Adrenal Gland', 'Thyroid', 'Lung', 'Spleen',
#        'Pancreas', 'Esophagus', 'Stomach', 'Colon', 'Small Intestine',
#        'Prostate', 'Testis', 'Nerve', 'Pituitary', 'Liver', 'Kidney',
#        'Cervix Uteri', 'Fallopian Tube', 'Bladder', 'Bone Marrow']


# for TISSUE in tissues:
#     params = {'sample_attributes': op.join(gtex_data_dir, 'GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'), 'tissue': TISSUE, 'transcript_data':op.join(gtex_data_dir, 'GTEx_Analysis_2017-06-05_v8_RSEMv1.3.0_transcript_tpm.gct')}
#     tissue_ids = retrieve_GTEX_tissue_sampleids(params['sample_attributes'], tissue=params['tissue'])
#     transcript_data = read_GTEX_transcript_expression(params['transcript_data'], tissue_ids)
#     transcript_data = clean_GTEX_tissue_transcript_counts(transcript_data, biomart)
#     transcript_data = variance_filtering(transcript_data)
#     transcript_data.to_csv(op.join(gtex_data_dir, f'{TISSUE}.tsv'), sep = '\t')

In [6]:
data_path = "/data/bionets/og86asub/alternet-project/alternet/data"
gtex_data_dir = '/data/bionets/datasets/hackathon/data/GTEX'

results_path_newa = "/data/bionets/og86asub/alternet-project/alternet/results-magnet/"
os.makedirs(results_path_newa, exist_ok = True)


# Reference files
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"


biomart = pd.read_csv(op.join(data_path, biomart_path), sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()

TISSUE = "DCM"

results_path = f"/data/bionets/og86asub/alternet-project/alternet/results-magnet/{TISSUE}/"

    
magnet_path = op.join(data_path, f"{TISSUE}_magnet_prefiltered_tpm.tsv")
transcript_data = pd.read_csv(magnet_path, sep='\t')


#transcript_data = pd.read_csv(op.join(gtex_data_dir, f'{TISSUE}.tsv'), sep = '\t', index_col = 0)

sample_cols = [c for c in transcript_data.columns if c not in ['transcript_id', 'gene_id']]

as_source_grn = pd.read_csv(op.join(results_path,  f"{TISSUE}_source_transcripts.tsv"), sep='\t')
as_source_grn = as_source_grn.rename(columns = {'source_transcript':'source', 'target_gene': 'target'})
as_source_grn = canonical_names(as_source_grn, tx2gene, gene2tx)
as_source_grn = filter_edges(as_source_grn)

# fully_as_aware = pd.read_csv(op.join(results_path,  f"{TISSUE}_fully_as_aware_raw.tsv"), sep='\t')
# fully_as_aware = fully_as_aware.rename(columns = {'source_transcript':'source', 'target_transcript': 'target'})
# fully_as_aware = canonical_names(fully_as_aware, tx2gene, gene2tx)
# fully_as_aware = filter_edges(fully_as_aware)


canonical_grn = pd.read_csv(op.join(results_path, f"{TISSUE}_source_genes.tsv"), sep='\t')
canonical_grn = canonical_grn.rename(columns = {'source_gene': 'source', 'target_gene': 'target'})
canonical_grn = canonical_names(canonical_grn, tx2gene, gene2tx)
canonical_grn = filter_edges(canonical_grn)



In [27]:
as_source_grn.reg_type.unique()

array(['TF', 'TF_SF'], dtype=object)

In [25]:
as_source_grn[(as_source_grn.reg_type == 'SF') & (as_source_grn.target_type=='transcript') & (as_source_grn.source_type == 'transcript') ]

,source,target,frequency,mean_importance,median_importance,source_type,target_type,source_gene,reg_type,target_gene,source_transcript,target_transcript


In [21]:
networks[(networks.reg_type == 'SF') & (networks.target_type=='transcript') ]

,source,target,frequency,mean_importance,median_importance,source_type,target_type,reg_type,source_gene,target_gene,...,target_transcript,edge_gg,edge_type,is_unique_edge,importance_ratio,category,n_equivalent_edge_types,reg_dominance,target_dominance,is_plausible
70672,ENSG00000003756,ENST00000196489,10,1.934000,1.597743,gene,transcript,SF,ENSG00000003756,ENSG00000083817,...,ENST00000196489,ENSG00000003756_ENSG00000083817,gene-transcript,False,1.000001,equivalent,2,0.895552,1.000000,True
70732,ENSG00000003756,ENST00000227503,10,3.870969,3.791944,gene,transcript,SF,ENSG00000003756,ENSG00000168066,...,ENST00000227503,ENSG00000003756_ENSG00000168066,gene-transcript,True,1.000001,specific_strict,0,0.895552,0.563203,True
70787,ENSG00000003756,ENST00000244061,10,4.590607,4.615313,gene,transcript,SF,ENSG00000003756,ENSG00000124226,...,ENST00000244061,ENSG00000003756_ENSG00000124226,gene-transcript,False,1.000001,equivalent,2,0.895552,1.000000,True
70830,ENSG00000003756,ENST00000253686,10,3.560869,3.871535,gene,transcript,SF,ENSG00000003756,ENSG00000131368,...,ENST00000253686,ENSG00000003756_ENSG00000131368,gene-transcript,True,1.000001,specific_strict,0,0.895552,0.648331,True
70840,ENSG00000003756,ENST00000254942,10,3.405590,3.005909,gene,transcript,SF,ENSG00000003756,ENSG00000132604,...,ENST00000254942,ENSG00000003756_ENSG00000132604,gene-transcript,True,1.000001,specific_strict,0,0.895552,0.618840,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32064951,ENSG00000270647,ENST00000632555,10,1.213799,1.300006,gene,transcript,SF,ENSG00000270647,ENSG00000265241,...,ENST00000632555,ENSG00000270647_ENSG00000265241,gene-transcript,True,1.000001,specific_strict,0,0.882591,0.576683,True
32064956,ENSG00000270647,ENST00000635721,10,1.001804,0.840935,gene,transcript,SF,ENSG00000270647,ENSG00000149136,...,ENST00000635721,ENSG00000270647_ENSG00000149136,gene-transcript,False,0.405493,other,0,0.882591,0.661307,True
32064987,ENSG00000270647,ENST00000642285,10,0.890477,0.856322,gene,transcript,SF,ENSG00000270647,ENSG00000168724,...,ENST00000642285,ENSG00000270647_ENSG00000168724,gene-transcript,True,1.000001,specific_strict,0,0.882591,0.589185,True
32065029,ENSG00000270647,ENST00000647488,10,7.480114,7.265354,gene,transcript,SF,ENSG00000270647,ENSG00000169554,...,ENST00000647488,ENSG00000270647_ENSG00000169554,gene-transcript,True,1.000001,specific_strict,0,0.882591,0.354302,True


In [10]:

networks = pd.concat([canonical_grn, as_source_grn])
networks = get_best_variable(networks)


gene_dominance, gene_n_isoforms, tx_expression_share  = compute_dominance_metrics(transcript_data, sample_cols)
networks = plausibility_filtering(networks, gene_dominance, r_dom = 0.9)

usage_df, reliability_df = calculate_transcript_usage(transcript_data)
transcript_data_temp = transcript_data.set_index('transcript_id')[sample_cols]
usage_df = usage_df.set_index('transcript_id')[sample_cols]

tf_net = networks[(networks.reg_type == 'TF')].copy()
# Decide if TF_SF is splice factor or transcription factor
nn = tf_sf_disambigouation_fully_as_aware(networks[(networks.reg_type == 'TF_SF') & (networks.target_type=='transcript')], regulator_list, transcript_data_temp, usage_df, reliability_df,
                                         RHO_MIN = 0.3,  Q_MIN = 0.05, DU_MIN = 0.1, n_cores = 16)


ab = compute_set_c(nn[nn.tfsf_category == 'tfsf_sf_like'], transcript_data, gene2tx, usage_df, reliability_df, sample_cols, epsilon=1e-6, n_cores=16)
ac = compute_set_c(networks[(networks.reg_type == 'SF') & (networks.target_type=='transcript') & (networks.source_type=='transcript')], transcript_data, gene2tx, usage_df, reliability_df, sample_cols, epsilon=1e-6, n_cores=16)
sf_net = pd.concat([ab, ac])


# Concatenate TF like regulators
ad = nn[nn.tfsf_category == 'tfsf_tf_like']
tf_net = pd.concat([ad, tf_net])
ambi_net = nn[nn.tfsf_category.isin(['tfsf_joint', 'tfsf_ambiguous'])]

AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA
AAAA


KeyError: 'target_transcript'

In [19]:
nn

,source,target,frequency,mean_importance,median_importance,source_type,target_type,reg_type,source_gene,target_gene,...,sf_rho,sf_pval,delta_usage,qc_ok,n_samples,tf_q,sf_q,tf_evidence,sf_evidence,tfsf_category
322,ENST00000245838,ENST00000357654,10,45.535619,45.040055,transcript,transcript,TF_SF,ENSG00000125676,ENSG00000012048,...,0.883769,5.526608e-54,3.556275e-01,True,160,8.890629e-01,3.229612e-52,False,True,tfsf_sf_like
4154,ENST00000447032,ENST00000651761,10,41.805158,42.830492,transcript,transcript,TF_SF,ENSG00000134453,ENSG00000143742,...,-0.553378,1.058978e-14,-4.879115e-01,True,166,1.325330e-27,2.885315e-14,True,True,tfsf_joint
2041,ENST00000360091,ENST00000446829,10,40.870438,38.701154,transcript,transcript,TF_SF,ENSG00000182944,ENSG00000112592,...,0.820109,4.028001e-41,8.540325e-01,True,164,7.171769e-17,7.242656e-40,True,True,tfsf_joint
5680,ENST00000598441,ENST00000290921,10,37.833816,38.256913,transcript,transcript,TF_SF,ENSG00000104852,ENSG00000159692,...,0.732496,3.459921e-29,5.700048e-01,True,166,2.595281e-27,2.469486e-28,True,True,tfsf_joint
3279,ENST00000406548,ENST00000564316,10,36.682786,36.552464,transcript,transcript,TF_SF,ENSG00000182944,ENSG00000103266,...,-0.664033,1.819848e-22,-3.052013e-01,True,166,7.295086e-08,8.273377e-22,True,True,tfsf_joint
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5826,ENST00000651767,ENST00000341105,10,1.101928,0.951631,transcript,transcript,TF_SF,ENSG00000102103,ENSG00000179348,...,0.347766,5.400279e-06,1.347643e-09,True,163,3.150154e-01,8.811973e-06,False,False,tfsf_ambiguous
3544,ENST00000421641,ENST00000373571,10,1.326746,0.951441,transcript,transcript,TF_SF,ENSG00000162613,ENSG00000147099,...,0.576411,4.429502e-16,5.436719e-03,True,166,9.610127e-02,1.320378e-15,False,False,tfsf_ambiguous
3134,ENST00000379888,ENST00000423823,10,1.080795,0.951224,transcript,transcript,TF_SF,ENSG00000134453,ENSG00000089335,...,0.294408,1.180188e-04,1.085221e-01,True,166,1.518566e-08,1.768391e-04,True,False,tfsf_tf_like
1685,ENST00000340281,ENST00000435416,10,1.045915,0.951021,transcript,transcript,TF_SF,ENSG00000162664,ENSG00000161298,...,0.494080,1.302175e-08,7.226834e-02,True,118,1.394986e-01,2.495799e-08,False,False,tfsf_ambiguous


In [9]:
transcript_data

,transcript_id,gene_id,GTEX-111CU-0426-SM-5GZY1,GTEX-111FC-1326-SM-5N9D9,GTEX-1122O-0526-SM-5N9DM,GTEX-117YX-2126-SM-5GIEL,GTEX-11DXX-0726-SM-5H12X,GTEX-11EQ9-0426-SM-5A5JY,GTEX-11LCK-0126-SM-5A5M5,GTEX-11NSD-0426-SM-5N9CR,...,GTEX-ZP4G-0326-SM-4YCEF,GTEX-ZPCL-0626-SM-DNZZ6,GTEX-ZPIC-0126-SM-DNZZ9,GTEX-ZPU1-1026-SM-4YCEQ,GTEX-ZTTD-1726-SM-57WEL,GTEX-ZV6S-0826-SM-5NQ6Z,GTEX-ZVP2-0526-SM-51MSC,GTEX-ZVZP-0726-SM-59HKA,GTEX-ZYFG-0726-SM-5GIDX,GTEX-ZZPU-0126-SM-5E446
13,ENST00000367770,ENSG00000000457,1.07,4.29,0.19,2.57,3.90,0.01,3.37,2.78,...,1.00,0.98,1.36,3.56,5.63,3.41,2.93,3.00,3.54,1.00
27,ENST00000374003,ENSG00000000938,0.39,1.43,3.07,0.89,0.96,1.71,2.01,2.72,...,0.76,1.96,1.67,3.75,0.75,1.22,1.45,0.61,0.74,0.34
28,ENST00000374004,ENSG00000000938,73.01,47.95,60.20,38.46,68.89,58.58,111.50,98.74,...,138.00,81.48,69.37,61.16,65.98,193.60,141.80,50.67,94.43,148.00
29,ENST00000374005,ENSG00000000938,17.77,7.03,19.97,6.86,13.92,22.36,24.25,17.31,...,34.13,21.72,17.84,13.71,6.20,28.22,30.68,13.97,15.25,32.03
30,ENST00000399173,ENSG00000000938,22.28,10.29,37.43,11.66,24.68,34.16,62.39,25.06,...,57.14,58.96,46.20,27.31,10.43,68.54,52.64,29.03,47.63,70.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198768,ENST00000638511,ENSG00000283703,4.73,7.13,9.44,1.33,1.43,0.64,7.20,5.86,...,7.55,0.80,29.54,8.99,0.10,0.78,3.75,2.33,2.34,3.17
198830,ENST00000640310,ENSG00000283787,14.17,48.42,35.74,21.30,33.47,14.98,38.06,26.92,...,16.57,22.84,19.12,21.37,22.48,18.41,23.15,16.95,26.74,15.13
199054,ENST00000535425,ENSG00000284194,3.88,2.28,4.33,0.00,6.73,0.50,2.63,2.00,...,11.87,2.63,1.20,6.16,7.59,2.27,10.48,2.21,7.46,2.60
199055,ENST00000543927,ENSG00000284194,1.98,2.65,2.65,3.33,0.00,0.76,0.55,4.38,...,3.01,1.61,3.08,0.52,5.99,2.18,1.13,1.62,1.78,2.66


In [19]:
splicefactor_targets = pd.read_csv('/data/bionets/og86asub/alternet-project/alternet/data/splicefactor_targets.tsv', sep = '\t', index_col=0)

In [20]:
for g in networks[networks.edge_type == 'gene-gene'].sort_values('mean_importance', ascending=False)['target'][0:500] :
    print(g)

ENSG00000102145
ENSG00000004939
ENSG00000105610
ENSG00000133742
ENSG00000169877
ENSG00000008438
ENSG00000029534
ENSG00000117228
ENSG00000092067
ENSG00000162894
ENSG00000112077
ENSG00000136929
ENSG00000004939
ENSG00000133742
ENSG00000136929
ENSG00000169877
ENSG00000148346
ENSG00000148346
ENSG00000101425
ENSG00000102145
ENSG00000160888
ENSG00000170476
ENSG00000124469
ENSG00000104267
ENSG00000104267
ENSG00000166598
ENSG00000008438
ENSG00000134285
ENSG00000112077
ENSG00000100219
ENSG00000224940
ENSG00000101336
ENSG00000011600
ENSG00000134827
ENSG00000169877
ENSG00000101425
ENSG00000256269
ENSG00000134827
ENSG00000117394
ENSG00000170476
ENSG00000134285
ENSG00000155660
ENSG00000138160
ENSG00000088325
ENSG00000148773
ENSG00000029534
ENSG00000206177
ENSG00000172232
ENSG00000145050
ENSG00000124469
ENSG00000100219
ENSG00000237649
ENSG00000120129
ENSG00000132470
ENSG00000123131
ENSG00000183508
ENSG00000115415
ENSG00000160255
ENSG00000156970
ENSG00000138496
ENSG00000110876
ENSG00000158869
ENSG0000

In [21]:
for g in networks[(networks.edge_type.isin(['transcript-transcript'])) & (networks.category.isin(['specific', 'specific_strict'])) & (networks.is_plausible)].sort_values('median_importance', ascending = False)['target_gene'][0:200]:
    print(g)

ENSG00000205571
ENSG00000236104
ENSG00000089280
ENSG00000149273
ENSG00000141367
ENSG00000134186
ENSG00000100162
ENSG00000099622
ENSG00000138326
ENSG00000100416
ENSG00000168056
ENSG00000105397
ENSG00000188536
ENSG00000102317
ENSG00000197530
ENSG00000142694
ENSG00000172354
ENSG00000100416
ENSG00000072110
ENSG00000122566
ENSG00000147854
ENSG00000156256
ENSG00000076108
ENSG00000196531
ENSG00000100416
ENSG00000108561
ENSG00000144029
ENSG00000244734
ENSG00000165280
ENSG00000204628
ENSG00000156261
ENSG00000145495
ENSG00000042980
ENSG00000096746
ENSG00000177600
ENSG00000153914
ENSG00000015133
ENSG00000173821
ENSG00000122545
ENSG00000258890
ENSG00000122224
ENSG00000172809
ENSG00000178719
ENSG00000149499
ENSG00000115053
ENSG00000073849
ENSG00000064687
ENSG00000138600
ENSG00000096384
ENSG00000269858
ENSG00000100813
ENSG00000108561
ENSG00000196510
ENSG00000141503
ENSG00000080824
ENSG00000100162
ENSG00000104904
ENSG00000131051
ENSG00000204256
ENSG00000156256
ENSG00000076928
ENSG00000163714
ENSG0000

In [22]:
def score_targets(edges, target_col, importance_col='median_importance'):
    """
    Compute target scores as weighted in-degree.
    score(target) = sum of importance for all edges targeting it
    """
    scores = edges.groupby(target_col)[importance_col].sum()
    return scores.sort_values(ascending=False)


In [23]:
def project_tx_to_gene(tx_scores, tx2gene, method='max'):
    """
    Project transcript-level scores to gene-level.
    
    Parameters:
    - tx_scores: pd.Series mapping transcript_id -> score
    - tx2gene: dict mapping transcript_id -> gene_id
    - method: 'max' (default) or 'sum'
    
    Returns:
    - gene_scores: pd.Series mapping gene_id -> score
    - rep_tx: dict mapping gene_id -> representative transcript
    """
    df = pd.DataFrame({
        'transcript_id': tx_scores.index,
        'score': tx_scores.values
    })
    df['gene_id'] = df['transcript_id'].map(tx2gene)
    df = df.dropna(subset=['gene_id'])
    
    if method == 'max':
        idx = df.groupby('gene_id')['score'].idxmax()
        result = df.loc[idx].set_index('gene_id')
        gene_scores = result['score'].sort_values(ascending=False)
        rep_tx = result['transcript_id'].to_dict()
    elif method == 'sum':
        gene_scores = df.groupby('gene_id')['score'].sum().sort_values(ascending=False)
        rep_tx = {}
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return gene_scores, rep_tx


In [24]:
def build_target_list(edges, target_col, importance_col='median_importance',
                      target_type='gene', tx2gene=None, gene2symbol=None,
                      K=500, projection_method='max'):
    """
    Build a ranked target list from an edge set.
    
    Parameters:
    - edges: DataFrame with edges
    - target_col: column containing target IDs
    - importance_col: column containing importance scores
    - target_type: 'gene' or 'transcript'
    - tx2gene: transcript to gene mapping (required if target_type='transcript')
    - gene2symbol: gene ID to symbol mapping
    - K: number of top genes to return
    - projection_method: 'max' or 'sum' (for transcript targets)
    
    Returns:
    - top_df: DataFrame with ranked targets
    - gene_symbols: list of gene symbols for g:Profiler
    - metadata: dict with scoring info
    """
    if len(edges) == 0:
        return pd.DataFrame(), [], {'n_edges': 0}
    
    # Score targets
    target_scores = score_targets(edges, target_col, importance_col)
    
    # Project if transcript targets
    if target_type == 'transcript':
        if tx2gene is None:
            raise ValueError("tx2gene mapping required for transcript targets")
        gene_scores, rep_tx = project_tx_to_gene(target_scores, tx2gene, method=projection_method)
    else:
        gene_scores = target_scores
        rep_tx = {}
    
    # Get top K
    top_genes = gene_scores.head(K)
    
    # Build result DataFrame
    top_df = pd.DataFrame({
        'rank': range(1, len(top_genes) + 1),
        'gene_id': top_genes.index,
        'score': top_genes.values
    })
    
    if gene2symbol is not None:
        top_df['gene_symbol'] = top_df['gene_id'].map(gene2symbol)
        top_df = top_df.dropna(subset=['gene_symbol'])
        gene_symbols = top_df['gene_symbol'].tolist()
    else:
        gene_symbols = top_df['gene_id'].tolist()
    
    if len(rep_tx) > 0:
        top_df['rep_transcript'] = top_df['gene_id'].map(rep_tx)
    
    metadata = {
        'n_edges': len(edges),
        'n_targets': len(target_scores),
        'n_genes': len(gene_scores),
        'n_top_symbols': len(gene_symbols),
        'projection_method': projection_method if target_type == 'transcript' else 'none'
    }
    
    return top_df, gene_symbols, metadata

In [25]:

tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2symbol = dict(zip(biomart['Gene stable ID'], biomart['Gene name']))
symbol2gene = dict(zip(biomart['Gene name'], biomart['Gene stable ID']))

In [26]:
TOP_K = 500
PROJECTION_METHOD = 'max'

In [27]:

# Build target list - targets are transcripts, project to genes
l3_top, l3_symbols, l3_meta = build_target_list(
    edges=networks[(networks.edge_type.isin(['transcript-transcript'])) & (networks.category.isin(['specific', 'specific_strict'])) & (networks.is_plausible)],
    target_col='target_transcript',
    importance_col='mean_importance',
    target_type='transcript',
    tx2gene=tx2gene,
    gene2symbol=gene2symbol,
    K=TOP_K,
    projection_method=PROJECTION_METHOD
)



In [28]:
for g in l3_top['gene_symbol']:
    print(g)

CTNNA1
RACK1
ANP32A
USP16
FLI1
CCT8
HNRNPA2B1
NCL
METTL9
MATR3
PPP3CB
TP53
HNRNPL
USP34
SMARCA2
ZNF655
MARK3
CLTC
CEP95
NACA
ARHGAP5
PPP4R3A
VCP
C11orf58
GTF3C1
HNRNPU
OAZ1
B2M
SLC15A3
YWHAZ
RPS3
HNRNPD
TRAPPC14
PMPCB
ITGA4
RPS21
H3-3A
HNRNPA1
FGD5
SEC11A
SYNE2
CLK4
USP25
CD44
ALDH3A2
HNRNPH3
RBM39
ANAPC5
PTP4A2
ARPC1B
UBA1
TTC14
GGA2
SLC25A39
TRIM27
EEF1D
MBP
YAF2
TRA2B
PLEKHO1
CLK1
RPL4
RPL31
BLNK
HNRNPH1
PTOV1
ATP5PF
TRMU
OXR1
RPL23A
CD53
CCNL2
RPL21
INTS3
ITGAE
CENPM
THOC1
NT5C
MCL1
MTREX
RASAL3
RPS24
EEF1B2
DBNL
ZNF580
SEPTIN7
SBNO2
PDE7A
PHF11
SH3D21
LENG8
SENP6
RPLP2
DHX15
MMS19
MRPS5
PCSK6
CHPT1
UBE2B
UBE2I
RNF130
SLC25A11
TAF15
RPL18
RNF146
NDUFA13
NELFCD
SGSM3
SELENOS
FNIP1
SLC25A3
PCTP
POLR2F
PIEZO1
PI4KA
TBC1D9B
PDXDC1
CRIP2
HDAC1
ANAPC7
MGAT1
PRPF3
PSMA3
SAP30BP
AP2M1
NAA10
HDAC6
TYK2
SP100
TCF7
PSMB10
ZNF76
EIF5
RPL32
SYNE1
NUP205
SETX
ARL6IP4
TRIP12
GLUL
E2F4
RAC1
SRSF3
WNK1
CLCN7
RSRP1
U2SURP
SCML1
RPL37A
SRSF10
ANXA7
CDK9
PPP2CA
MAP3K7
WDR59
N4BP2L2
CD151
SCAF8
GNB2
SA